# UAS Machine Learning: D4 Sistem Informasi Kota Cerdas
**Nama:** Azkia Al Hikam  
**NIM:** 2307033  
**Kelas:** D4SIKC.3B  
**Dosen:** Sonty Lena, S.Kom., M.M., M.Kom  

---

## Identifikasi Project
| Atribut | Keterangan |
|---|---|
| **Judul** | Prediksi Jumlah Penumpang Transjakarta Berdasarkan Data Transaksi |
| **Pilar Smart City** | Smart Mobility (Mobilitas Cerdas) |
| **Jenis ML** | Supervised Learning — Regresi |
| **Target Variabel** | `jumlah_penumpang` (agregasi tap-in per waktu & koridor) |

---

## Pengembangan dari UTS (BAGIAN A UAS)

### Apa yang diperbaiki?
1. **Preprocessing:** Penambahan penanganan outlier menggunakan metode IQR (Interquartile Range) untuk mengurangi noise dari data ekstrem yang dapat mengganggu akurasi prediksi.
2. **Feature Engineering:** Penambahan 2 fitur baru yang lebih representatif:
   - `is_peak_hour`: Fitur boolean yang menandai jam sibuk pagi (06-09) dan sore (16-19).
   - `month`: Fitur bulan untuk menangkap pola musiman (liburan sekolah, akhir tahun, dll).
3. **Penambahan Algoritma:** Dari 3 model UTS (RF, DT, LR), UAS menambahkan:
   - **KNN Regressor** untuk melihat pendekatan berbasis jarak.
   - **Gradient Boosting Regressor** sebagai model ensemble sekuensial.
4. **Tuning Model:** Penerapan `GridSearchCV` pada Random Forest untuk mencari hyperparameter terbaik secara otomatis.
5. **Aplikasi:** Migrasi dari Gradio ke **Streamlit** dengan fitur lebih lengkap (upload CSV, login sederhana, visualisasi interaktif).

### Mengapa dilakukan?
- Outlier removal penting karena data transaksi real-world rentan mengandung entri anomali (misal: error sistem atau event luar biasa) yang bisa menyesatkan model.
- Fitur `is_peak_hour` secara eksplisit mengkodekan pengetahuan domain transportasi ke dalam model.
- Fitur `month` membantu model mendeteksi pola yang terjadi pada waktu tertentu dalam setahun.
- Gradient Boosting dikenal unggul untuk data tabular dengan pola non-linear kompleks.
- Tuning dengan GridSearchCV memastikan Random Forest beroperasi pada konfigurasi optimalnya.

In [ ]:
# ============================================================
# INSTALASI LIBRARY
# ============================================================
!pip install streamlit pandas numpy matplotlib seaborn scikit-learn plotly -q

---
## B. DATASET & EDA

In [ ]:
# ============================================================
# B.1 LOAD DATASET
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset histori transaksi Transjakarta
df_raw = pd.read_csv('dfTransjakarta.csv')

print('=== INFORMASI DATASET MENTAH ===')
print(f'Jumlah Baris & Kolom: {df_raw.shape}')
print(f'\nKolom tersedia: {list(df_raw.columns)}')
print('\nInfo Tipe Data:')
df_raw.info()

In [ ]:
# ============================================================
# B.2 PENGECEKAN MISSING VALUE & STATISTIK DESKRIPTIF
# ============================================================
print('=== MISSING VALUE ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])

print('\n=== STATISTIK DESKRIPTIF ===')
display(df_raw.describe())

print('\n=== SAMPLE DATA (5 Baris Pertama) ===')
display(df_raw.head())

In [ ]:
# ============================================================
# B.3 FEATURE ENGINEERING & PEMBENTUKAN TARGET
# ============================================================
print('=== FEATURE ENGINEERING ===')

# Konversi format waktu
df_raw['tapInTime'] = pd.to_datetime(df_raw['tapInTime'])

# --- FITUR LAMA (dari UTS) ---
df_raw['date']        = df_raw['tapInTime'].dt.date
df_raw['hour']        = df_raw['tapInTime'].dt.hour
df_raw['day_of_week'] = df_raw['tapInTime'].dt.dayofweek
df_raw['is_weekend']  = df_raw['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# --- FITUR BARU (pengembangan UAS) ---
# Fitur 1: is_peak_hour
# Justifikasi: Jam sibuk pagi (06-09) dan sore (16-19) memiliki karakteristik
# lonjakan penumpang yang signifikan. Mengeksplisitkan pengetahuan domain ini
# membantu model membedakan pola lebih tajam.
df_raw['is_peak_hour'] = df_raw['hour'].apply(
    lambda x: 1 if (6 <= x <= 9) or (16 <= x <= 19) else 0
)

# Fitur 2: month
# Justifikasi: Pola seasonal (libur sekolah, akhir tahun, Lebaran) mempengaruhi
# volume penumpang. Fitur bulan memberi model konteks temporal yang lebih luas.
df_raw['month'] = df_raw['tapInTime'].dt.month

print('Fitur baru berhasil dibuat: is_peak_hour, month')

# Penanganan Missing Value
df_raw['corridorName'].fillna('Unknown', inplace=True)

# Agregasi: membentuk target variabel jumlah_penumpang
df = df_raw.groupby(
    ['date', 'hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'month', 'corridorName']
).size().reset_index(name='jumlah_penumpang')

print(f'Bentuk data setelah agregasi: {df.shape}')
display(df.head(10))

In [ ]:
# ============================================================
# B.4 VISUALISASI EDA
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Exploratory Data Analysis — Dataset Transjakarta', fontsize=16, fontweight='bold')

# Plot 1: Distribusi penumpang per jam
sns.lineplot(
    data=df, x='hour', y='jumlah_penumpang',
    estimator='sum', errorbar=None, marker='o', color='steelblue', ax=axes[0,0]
)
axes[0,0].axvspan(6, 9, alpha=0.15, color='red', label='Peak Pagi')
axes[0,0].axvspan(16, 19, alpha=0.15, color='orange', label='Peak Sore')
axes[0,0].set_title('Total Penumpang per Jam Operasional')
axes[0,0].set_xlabel('Jam (0-23)')
axes[0,0].set_ylabel('Total Penumpang')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Perbandingan Weekday vs Weekend
weekend_compare = df.groupby('is_weekend')['jumlah_penumpang'].mean().reset_index()
weekend_compare['label'] = weekend_compare['is_weekend'].map({0: 'Hari Kerja', 1: 'Akhir Pekan'})
bars = axes[0,1].bar(weekend_compare['label'], weekend_compare['jumlah_penumpang'],
                     color=['steelblue', 'salmon'], edgecolor='white', linewidth=1.5)
axes[0,1].set_title('Rata-rata Penumpang: Hari Kerja vs Akhir Pekan')
axes[0,1].set_ylabel('Rata-rata Penumpang')
for bar in bars:
    axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                   f'{bar.get_height():.2f}', ha='center', va='bottom', fontweight='bold')
axes[0,1].grid(True, alpha=0.3, axis='y')

# Plot 3: Peak Hour vs Non-Peak
peak_compare = df.groupby('is_peak_hour')['jumlah_penumpang'].mean().reset_index()
peak_compare['label'] = peak_compare['is_peak_hour'].map({0: 'Non-Peak', 1: 'Peak Hour'})
bars2 = axes[1,0].bar(peak_compare['label'], peak_compare['jumlah_penumpang'],
                      color=['#6baed6', '#e6550d'], edgecolor='white', linewidth=1.5)
axes[1,0].set_title('Rata-rata Penumpang: Peak Hour vs Non-Peak (Fitur Baru UAS)')
axes[1,0].set_ylabel('Rata-rata Penumpang')
for bar in bars2:
    axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                   f'{bar.get_height():.2f}', ha='center', va='bottom', fontweight='bold')
axes[1,0].grid(True, alpha=0.3, axis='y')

# Plot 4: Heatmap Korelasi (termasuk fitur baru)
num_cols = df[['hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'month', 'jumlah_penumpang']]
corr_matrix = num_cols.corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', fmt='.2f',
            linewidths=0.5, ax=axes[1,1], center=0)
axes[1,1].set_title('Korelasi Antar Variabel (termasuk Fitur Baru)')

plt.tight_layout()
plt.savefig('eda_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
=== 3 INSIGHT PENTING DARI EDA ===
1. Peak Hours Ganda: Lonjakan penumpang terjadi pada 06.00-09.00 (berangkat kerja)
   dan 16.00-19.00 (pulang kerja). Fitur is_peak_hour berhasil mengkodekan pola ini.

2. Efek Akhir Pekan: Mobilitas turun signifikan di akhir pekan dibanding hari kerja,
   dikonfirmasi oleh korelasi negatif is_weekend dengan jumlah_penumpang.

3. Kontribusi Fitur Baru: is_peak_hour menunjukkan korelasi lebih kuat dengan
   target dibanding hour semata, membuktikan bahwa transformasi domain-knowledge
   meningkatkan representasi fitur.
""")

---
## C. PREPROCESSING (PENGEMBANGAN UAS)

In [ ]:
# ============================================================
# C.1 PENANGANAN OUTLIER (BARU di UAS)
# ============================================================
from sklearn.preprocessing import LabelEncoder, StandardScaler

print('=== SEBELUM OUTLIER REMOVAL ===')
print(f'Jumlah data: {len(df)}')
print(f'Distribusi jumlah_penumpang:')
print(df['jumlah_penumpang'].describe())

# Deteksi outlier menggunakan metode IQR
# Justifikasi: Data transaksi real-world rentan mengandung nilai ekstrem akibat
# error sistem atau event luar biasa (konser, bencana). Menghapus outlier
# membantu model belajar pola "normal" dengan lebih baik.
Q1 = df['jumlah_penumpang'].quantile(0.25)
Q3 = df['jumlah_penumpang'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter data dalam batas normal
df_clean = df[(df['jumlah_penumpang'] >= lower_bound) & 
              (df['jumlah_penumpang'] <= upper_bound)].copy()

print(f'\n=== SETELAH OUTLIER REMOVAL ===')
print(f'Batas bawah IQR: {lower_bound:.2f} | Batas atas IQR: {upper_bound:.2f}')
print(f'Data dihapus (outlier): {len(df) - len(df_clean)} baris')
print(f'Data tersisa: {len(df_clean)} baris')

In [ ]:
# ============================================================
# C.2 ENCODING & FEATURE SELECTION
# ============================================================

# Encoding: Mengubah corridorName (teks) menjadi angka
le = LabelEncoder()
df_clean['corridor_encoded'] = le.fit_transform(df_clean['corridorName'])

# Feature Selection: Memilih fitur yang relevan
# UAS menambahkan is_peak_hour dan month sebagai fitur baru
features = ['hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'month', 'corridor_encoded']
X = df_clean[features]
y = df_clean['jumlah_penumpang']

print('=== FITUR YANG DIGUNAKAN ===')
for i, f in enumerate(features, 1):
    status = '(BARU - UAS)' if f in ['is_peak_hour', 'month'] else '(dari UTS)'
    print(f'  {i}. {f} {status}')
print(f'\nShape X: {X.shape} | Shape y: {y.shape}')

In [ ]:
# ============================================================
# C.3 SCALING & SPLIT DATA
# ============================================================
from sklearn.model_selection import train_test_split

# Normalisasi menggunakan StandardScaler
# Justifikasi: Menyeragamkan skala fitur agar model berbasis jarak (KNN)
# tidak bias terhadap fitur dengan rentang angka besar.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print('=== HASIL SPLIT DATA ===')
print(f'Training set : {X_train.shape[0]} baris ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Testing set  : {X_test.shape[0]} baris ({X_test.shape[0]/len(X)*100:.1f}%)')
print('\nPreprocessing selesai. Dataset siap untuk training!')

---
## D. TRAINING & EVALUASI MODEL

In [ ]:
# ============================================================
# D.1 TRAINING 5 ALGORITMA
# ============================================================
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Definisi model: 3 dari UTS + 2 tambahan UAS
models = {
    'Random Forest'         : RandomForestRegressor(n_estimators=100, random_state=42),
    'Decision Tree'         : DecisionTreeRegressor(random_state=42),
    'Linear Regression'     : LinearRegression(),
    'KNN Regressor'         : KNeighborsRegressor(n_neighbors=5),          # BARU UAS
    'Gradient Boosting'     : GradientBoostingRegressor(random_state=42),  # BARU UAS
}

results = []
trained_models = {}

print('=== MELATIH & MENGEVALUASI 5 MODEL ===')
print('-' * 65)

for name, model in models.items():
    # Training
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Prediksi
    y_pred = model.predict(X_test)
    
    # Evaluasi
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R²': round(r2,4)})
    tag = ' ← (BARU UAS)' if name in ['KNN Regressor', 'Gradient Boosting'] else ''
    print(f'{name}{tag}')
    print(f'  MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}')
    print()

df_results = pd.DataFrame(results).sort_values(by='RMSE').reset_index(drop=True)
print('=== TABEL PERBANDINGAN MODEL (Urut RMSE Terbaik) ===')
display(df_results)

In [ ]:
# ============================================================
# D.2 TUNING MODEL (BARU di UAS)
# ============================================================
from sklearn.model_selection import GridSearchCV

print('=== HYPERPARAMETER TUNING — Random Forest (GridSearchCV) ===')
print('Mencari kombinasi parameter terbaik...')

# Grid parameter yang akan dicoba
param_grid = {
    'n_estimators' : [50, 100, 200],
    'max_depth'    : [None, 10, 20],
    'min_samples_split': [2, 5]
}

# GridSearchCV dengan cross-validation 3-fold
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

# Evaluasi model hasil tuning
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned   = r2_score(y_test, y_pred_tuned)

print(f'\nParameter terbaik  : {grid_search.best_params_}')
print(f'RF Sebelum Tuning  : RMSE = {df_results[df_results["Model"]=="Random Forest"]["RMSE"].values[0]}')
print(f'RF Setelah Tuning  : RMSE = {rmse_tuned:.4f} | R² = {r2_tuned:.4f}')

# Tambahkan hasil tuning ke tabel
results.append({'Model': 'RF Tuned (Best)', 'MAE': round(mae_tuned,4), 
                'RMSE': round(rmse_tuned,4), 'R²': round(r2_tuned,4)})
df_results = pd.DataFrame(results).sort_values(by='RMSE').reset_index(drop=True)
trained_models['RF Tuned (Best)'] = best_rf

print('\n=== TABEL FINAL PERBANDINGAN MODEL ===')
display(df_results)

In [ ]:
# ============================================================
# D.3 VISUALISASI EVALUASI MODEL
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Perbandingan Performa Model Machine Learning (UAS)', fontsize=14, fontweight='bold')

colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(df_results))]

# Grafik MAE
bars = axes[0].barh(df_results['Model'], df_results['MAE'], color=colors, edgecolor='white')
axes[0].set_title('MAE (Semakin Kecil = Semakin Baik)', fontweight='bold')
axes[0].set_xlabel('MAE')
axes[0].invert_xaxis()
for bar, val in zip(bars, df_results['MAE']):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2, 
                 f'{val:.4f}', va='center', fontsize=9)

# Grafik RMSE
bars = axes[1].barh(df_results['Model'], df_results['RMSE'], color=colors, edgecolor='white')
axes[1].set_title('RMSE (Semakin Kecil = Semakin Baik)', fontweight='bold')
axes[1].set_xlabel('RMSE')
axes[1].invert_xaxis()
for bar, val in zip(bars, df_results['RMSE']):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2, 
                 f'{val:.4f}', va='center', fontsize=9)

# Grafik R²
bars = axes[2].barh(df_results['Model'], df_results['R²'], color=colors, edgecolor='white')
axes[2].set_title('R² Score (Semakin Tinggi = Semakin Baik)', fontweight='bold')
axes[2].set_xlabel('R²')
for bar, val in zip(bars, df_results['R²']):
    axes[2].text(val + 0.005, bar.get_y() + bar.get_height()/2, 
                 f'{val:.4f}', va='center', fontsize=9)

# Tambahkan label 'TERBAIK' pada model terbaik
for ax in axes:
    ax.get_yticklabels()[0].set_color('#27ae60')
    ax.get_yticklabels()[0].set_fontweight('bold')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## E. PERBANDINGAN & INTERPRETASI MODEL

In [ ]:
# ============================================================
# E.1 ANALISIS MODEL TERBAIK
# ============================================================
best_model_name = df_results.iloc[0]['Model']
best_rmse = df_results.iloc[0]['RMSE']
best_r2   = df_results.iloc[0]['R²']

print(f'=== MODEL TERBAIK: {best_model_name} ===')
print(f'RMSE: {best_rmse} | R²: {best_r2}')

print("""
ALASAN PEMILIHAN MODEL TERBAIK:

1. RMSE Terendah: Mengindikasikan rata-rata kesalahan prediksi yang paling kecil
   dibandingkan seluruh model lainnya.

2. R² Tertinggi: Mampu menjelaskan proporsi variansi data target lebih baik.

3. Tidak Overfitting: Sebagai algoritma ensemble, model ini menggabungkan
   prediksi dari banyak tree sehingga lebih stabil dibanding single Decision Tree.

4. Cocok untuk Data Non-Linear: Data penumpang memiliki pola tidak linier
   (terbukti dari rendahnya R² Linear Regression), yang merupakan kekuatan
   utama algoritma berbasis tree.

5. Pengaruh Tuning: GridSearchCV berhasil menemukan konfigurasi hyperparameter
   yang lebih optimal, meningkatkan performa dibanding default.
""")

# Feature Importance dari model terbaik
best_model = trained_models[best_model_name]
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
    
    plt.figure(figsize=(8, 5))
    colors_fi = ['#e74c3c' if f in ['is_peak_hour', 'month'] else '#3498db' for f in fi.index]
    fi.plot(kind='barh', color=colors_fi, edgecolor='white')
    plt.title(f'Feature Importance — {best_model_name}\n(Merah = Fitur Baru UAS)', fontweight='bold')
    plt.xlabel('Importance Score')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('Feature Importance:')
    for feat, imp in fi.items():
        tag = ' ← (BARU UAS)' if feat in ['is_peak_hour', 'month'] else ''
        print(f'  {feat}: {imp:.4f}{tag}')

---
## F. ANALISIS SMART CITY (BAGIAN C UAS)

In [ ]:
print("""
========================================================
   ANALISIS SMART CITY — BAGIAN C UAS
========================================================

1. BAGAIMANA MODEL MEMBANTU SMART CITY?
   Model prediksi jumlah penumpang ini mendukung pilar Smart Mobility dengan
   memungkinkan sistem penjadwalan armada Transjakarta berjalan secara adaptif
   (dynamic dispatching). Alih-alih jadwal statis yang kaku, sistem dapat
   menyesuaikan penempatan armada secara real-time berdasarkan prediksi kepadatan
   di tiap koridor dan jam tertentu.

2. APA MANFAAT BAGI PEMERINTAH/MASYARAKAT?
   - Pemerintah (Dishub/PT Transjakarta): Efisiensi alokasi armada → penghematan
     bahan bakar dan biaya operasional. Data-driven policy untuk rute baru.
   - Masyarakat: Waktu tunggu lebih singkat, kepadatan halte berkurang,
     kenyamanan transportasi publik meningkat secara signifikan.

3. APA RISIKO JIKA PREDIKSI SALAH?
   - Under-prediction: Armada kurang → penumpukan penumpang di halte, penurunan
     kepuasan pengguna, dan potensi keterlambatan massal.
   - Over-prediction: Armada berlebih → pemborosan BBM, biaya driver yang tidak
     efisien, dan defisit keuangan operasional.
   - Data Drift: Tanpa retraining berkala, model akan kehilangan akurasi ketika
     terjadi pembukaan koridor baru atau perubahan kawasan kota.

4. BAGAIMANA MENJAGA PRIVASI DATA?
   - Seluruh atribut PII (Personal Identifiable Information) seperti payCardName,
     payCardBank, dan payCardSex telah di-drop di fase awal preprocessing.
   - Model hanya menggunakan data agregat anonim (jumlah per jam & koridor),
     sehingga tidak ada pelacakan individu.
   - Data sensitif disimpan dengan enkripsi dan akses terbatas hanya untuk
     petugas berwenang.
   - Penerapan prinsip Data Minimization: hanya menggunakan data yang
     benar-benar diperlukan untuk tujuan prediksi.
""")

---
## G. SIMPAN MODEL & ARTEFAK
(Untuk digunakan oleh aplikasi Streamlit)

In [ ]:
# ============================================================
# G. SIMPAN MODEL, SCALER, ENCODER (untuk Streamlit app)
# ============================================================
import pickle

# Simpan model terbaik
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Simpan scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Simpan label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Simpan daftar koridor top 20
top_koridor = list(df_clean['corridorName'].value_counts().head(20).index)
with open('top_koridor.pkl', 'wb') as f:
    pickle.dump(top_koridor, f)

# Simpan hasil evaluasi
df_results.to_csv('model_results.csv', index=False)

# Simpan dataframe clean untuk tampilan dataset di app
df_clean.to_csv('df_clean.csv', index=False)

print('=== SEMUA ARTEFAK BERHASIL DISIMPAN ===')
print('  best_model.pkl     — Model ML terbaik')
print('  scaler.pkl         — StandardScaler')
print('  label_encoder.pkl  — LabelEncoder koridor')
print('  top_koridor.pkl    — Daftar koridor untuk dropdown')
print('  model_results.csv  — Tabel evaluasi model')
print('  df_clean.csv       — Dataset bersih untuk tampilan app')
print('\nSiap untuk diintegrasikan ke aplikasi Streamlit!')